In [3]:
#  IS211_Assignment 2
# datetime,  logging, urllib, random, argparse, pprint
#First import the libraries from course content. URLLIB2 doesnt exist in Python 3. Jupyer notebook is in Python3. Modern URLLIB.Request was used.
import argparse
import csv
import datetime
import io
import logging
import urllib.request

In [4]:
def downloadData(url):
    with urllib.request.urlopen(url) as response:
        return response.read()

In [5]:
url = "https://s3.amazonaws.com/cuny-is211-spring2015/birthdays100.csv"

#this is the URL link from the Assignment 2.

In [6]:
raw_bytes = downloadData(url)
csv_text = raw_bytes.decode("utf-8")
# USe the Download DATA function from standard library to get raw data. Then use decode to turn bytes into strings.
# I read the tutorial to understand bytes and strings.https://realpython.com/convert-python-bytes-to-strings/ 
#The easiest way was to convert to CSV

csv_lines = csv_text.strip().splitlines()
reader = csv.reader(csv_lines)
#splitlines splits a string into lines and .split removes blank spaces infront and after. CSV reader reads CSV file and prints.
data_rows = list(reader)

for row in data_rows[:5]:
    print(row)

['id', 'name', 'birthday']
['1', 'Charles Paige', '06/01/1963']
['2', 'Andrew Bell', '29/03/1972']
['3', 'Charles Reid', '14/06/2009']
['4', 'Sebastian Hudson', '15/08/1991']


In [14]:
def processData(file_contents):
    logger = logging.getLogger("assignment2")
    person_dict = {}
#logger is used. 
    if isinstance(file_contents, bytes):
        file_contents = file_contents.decode("utf-8")

    lines = file_contents.strip().splitlines()
    reader = csv.reader(lines)
#same as before to get CSV from Bytes from URL.
    for line_num, row in enumerate(reader, start=1):
        if not row:
            continue

        person_id, name, birthday_str = row[0], row[1], row[2]

        try:
            birthday_date = datetime.datetime.strptime(
                birthday_str, "%d/%m/%Y"
            )
            person_dict[int(person_id)] = (name, birthday_date)
        except ValueError:
            logger.error(
                f"Error processing line #{line_num} for ID #{person_id}"
            )
#lets us create a logger to see what is process ok and what is error.
    return person_dict

In [15]:
# 4. Execute and PRINT results
url = "https://s3.amazonaws.com/cuny-is211-spring2015/birthdays100.csv"
csv_data = downloadData(url)
results = processData(csv_data)

# Print total entries processed
print(f"Total valid records processed: {len(results)}\n")

Error processing line #1 for ID #id
Error processing line #14 for ID #13
Error processing line #28 for ID #27
Error processing line #31 for ID #30
Error processing line #48 for ID #47
Error processing line #49 for ID #48
Error processing line #68 for ID #67
Error processing line #72 for ID #71
Error processing line #79 for ID #78
Error processing line #88 for ID #87
Error processing line #94 for ID #93


Total valid records processed: 90



In [16]:
#e purpose of this function is to print the name and birthday of a given user identified by the input id. 

def displayPerson(id, personData):
    if id in personData:
        name, birthday = personData[id]
        formatted_date = birthday.strftime("%Y-%m-%d")
        print(f"Person #{id} is {name} with a birthday of {formatted_date}")
    else:
        print("No user found with that id")

In [17]:
# Test with an existing ID (e.g., ID 1)
displayPerson(1, results)

Person #1 is Charles Paige with a birthday of 1963-01-06


In [18]:
displayPerson(999, results)

No user found with that id


In [19]:
#PArt 5

import argparse
import sys

# Define the argument parser
parser = argparse.ArgumentParser(
    description="Download and process CSV birthday data."
)

# Add required --url argument
parser.add_argument(
    "--url", type=str, required=True, help="URL pointing to the CSV dataset"
)

# In Jupyter, we simulate command line arguments by passing a list to parse_args()
test_args = [
    "--url",
    "https://s3.amazonaws.com/cuny-is211-spring2015/birthdays100.csv",
]

# Parse arguments
args = parser.parse_args(test_args)

# Retrieve the url parameter
url = args.url
print(f"URL successfully received: {url}")

URL successfully received: https://s3.amazonaws.com/cuny-is211-spring2015/birthdays100.csv


In [20]:
import urllib.request


def downloadData(url):
    with urllib.request.urlopen(url) as response:
        return response.read()


# Execute downloadData with error handling
try:
    csvData = downloadData(url)
    print("Data downloaded successfully!")
    print(f"Downloaded byte size: {len(csvData)} bytes")

except Exception as e:
    print(
        f"ERROR: Failed to download data from the provided URL. Details: {e}"
    )
    sys.exit(1)

Data downloaded successfully!
Downloaded byte size: 2776 bytes


In [21]:
import logging

# Configure logger named 'assignment2'
logger = logging.getLogger("assignment2")
logger.setLevel(logging.ERROR)

# Create a file handler to log messages to 'errors.log'
file_handler = logging.FileHandler("errors.log", mode="w")

# Optional: Format log entries with timestamps
formatter = logging.Formatter("%(asctime)s - %(levelname)s - %(message)s")
file_handler.setFormatter(formatter)

# Clear any existing handlers to prevent duplicate lines if run multiple times in Jupyter
if logger.hasHandlers():
    logger.handlers.clear()

logger.addHandler(file_handler)
print("Logger 'assignment2' configured to write to errors.log")

Logger 'assignment2' configured to write to errors.log


In [22]:
import csv
import datetime


def processData(file_contents):
    logger = logging.getLogger("assignment2")
    person_dict = {}

    if isinstance(file_contents, bytes):
        file_contents = file_contents.decode("utf-8")

    lines = file_contents.strip().splitlines()
    reader = csv.reader(lines)

    for line_num, row in enumerate(reader, start=1):
        if not row:
            continue

        person_id, name, birthday_str = row[0], row[1], row[2]

        try:
            birthday_date = datetime.datetime.strptime(
                birthday_str, "%d/%m/%Y"
            )
            person_dict[int(person_id)] = (name, birthday_date)
        except (ValueError, IndexError):
            logger.error(
                f"Error processing line #{line_num} for ID #{person_id}"
            )

    return person_dict


# Pass csvData to processData and save in personData
personData = processData(csvData)

# Verify the result
print(f"Successfully processed {len(personData)} entries into personData.")

# Preview the first entry in personData
first_key = list(personData.keys())[0]
print(f"Sample Entry (ID {first_key}): {personData[first_key]}")

Successfully processed 90 entries into personData.
Sample Entry (ID 1): ('Charles Paige', datetime.datetime(1963, 1, 6, 0, 0))


In [23]:
with open("errors.log", "r") as f:
    print(f.read())

2026-09-14 22:45:11,449 - ERROR - Error processing line #1 for ID #id
2026-09-14 22:45:11,451 - ERROR - Error processing line #14 for ID #13
2026-09-14 22:45:11,451 - ERROR - Error processing line #28 for ID #27
2026-09-14 22:45:11,452 - ERROR - Error processing line #31 for ID #30
2026-09-14 22:45:11,452 - ERROR - Error processing line #48 for ID #47
2026-09-14 22:45:11,452 - ERROR - Error processing line #49 for ID #48
2026-09-14 22:45:11,452 - ERROR - Error processing line #68 for ID #67
2026-09-14 22:45:11,453 - ERROR - Error processing line #72 for ID #71
2026-09-14 22:45:11,453 - ERROR - Error processing line #79 for ID #78
2026-09-14 22:45:11,453 - ERROR - Error processing line #88 for ID #87
2026-09-14 22:45:11,453 - ERROR - Error processing line #94 for ID #93



In [ ]:
# -------------------------------------------------------------------
# Function: displayPerson
# -------------------------------------------------------------------
def displayPerson(id, personData):
    if id in personData:
        name, birthday = personData[id]
        formatted_date = birthday.strftime("%Y-%m-%d")
        print(f"Person #{id} is {name} with a birthday of {formatted_date}")
    else:
        print("No user found with that id")


# -------------------------------------------------------------------
# Step 5 Prompt Loop
# -------------------------------------------------------------------
while True:
    try:
        user_input = int(
            input(
                "\nEnter a person ID to lookup (enter 0 or negative number to exit): "
            )
        )

        if user_input <= 0:
            print("Exiting program.")
            break

        displayPerson(user_input, personData)

    except ValueError:
        print("Invalid input. Please enter an integer.")


Enter a person ID to lookup (enter 0 or negative number to exit):  2


Person #2 is Andrew Bell with a birthday of 1972-03-29
